In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from natsort import natsorted
import numpy as np
import re
import sys

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(rf'../../../../tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

tidy3dAPI = os.environ["API_TIDY3D_KEY"]




In [2]:
dir = rf"./data/diffraction_monitor_data"
os.makedirs(dir, exist_ok=True)

In [3]:
try:
    data_path = f"{dir}/20260708_average_diffraction_n_3.4_ff_0.237_ffh_0.185_schulz.h5"
    data_old = AM.read_hdf5_as_dict(data_path)
    print(data_old.keys())
except FileNotFoundError:
    print("File not found.")
    data_old = {}
except Exception as e:
    print(f"An error occurred: {e}")
    data_old = {}

folder_path = rf"../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction"

Error reading HDF5 file: [Errno 2] Unable to synchronously open file (unable to open file: name = './data/diffraction_monitor_data/20260708_average_diffraction_n_3.4_ff_0.237_ffh_0.185_schulz.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)
dict_keys([])


In [4]:
reference_object = AM.loadFromFile(key = tidy3dAPI, file_path=os.path.join(folder_path, "reference.txt"),get_ref=False)

amps_ref = reference_object.sim_data["diffraction"].amps
Pinc = np.abs(amps_ref.sel(orders_x=0, orders_y=0, polarization="p"))**2

reference_exit = reference_object.sim_data["flux1"].flux


Configured successfully.


16:48:08 W. Europe Daylight Time Billed flex credit cost: 0.109.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [5]:
n_values = list(data_old['n_values'] )if 'n_values' in data_old.keys() else []
ff_values =list( data_old['ff']) if 'ff' in data_old.keys() else []
size_values = list(data_old['sizes'])if 'sizes' in data_old.keys() else []
z_values = list(data_old['z_values']) if 'z_values' in data_old.keys() else []
values = data_old['transmission_data'] if 'transmission_data' in data_old.keys() else {}
reference_entry=None

# Polar/azimuthal angle of every diffraction order, dims (orders_x, orders_y, f).
# These depend only on the transverse period, the Bloch vector, the frequency list and
# the medium at the monitor -- none of which vary in this sweep (only the slab thickness
# changes, and that is along z). So capture them once and check the rest agree.
class DiffractionGeometryError(RuntimeError):
    """Raised when a sim's diffraction grid differs from the one the angle mask was built on."""

theta_orders = None
phi_orders = None
diffraction_sim_size = None
_angle_geometry = None

# Loop through all files in the folder
for dirpath, dirnames, filenames in os.walk(folder_path):
      try:
        z_value = float(re.search(r'z_([+-]?\d+(?:\.\d+)?)', dirpath).group(1))
      except AttributeError:
        print(f"Could not extract z_value from directory: {dirpath}")
        continue
      
      z_values.append(z_value)
      for filename in filenames:
        try:
            n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff = float(re.search(r'ffr_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            size = float(re.search(r'size_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            sample = float(re.search(r'sample_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff_values.append(ff)
            n_values.append(n_value)
            size_values.append(size)
            try:
                test_val = values[n_value][ff][z_value][size][sample]
                print(f"Data for n={n_value}, ff={ff}, z={z_value}, size={size}, sample={sample} already exists. Skipping file: {filename}")
            except KeyError:
                #Retrieve simulation data 
                if os.path.isfile(os.path.join(dirpath, filename)):
                  file=os.path.join(dirpath, filename)
                  structure_1 = AM.loadFromFile(key = tidy3dAPI, file_path=file,get_ref=False)
                  sim_data_i = structure_1.sim_data
                  transmission_entry = sim_data_i['flux2'].flux
                  transmission_exit = sim_data_i['flux1'].flux

                  #Diffraction-order angles: identical for every sim, so store the first set
                  diffraction_i = sim_data_i["diffraction"]
                  geometry_i = (tuple(np.ravel(diffraction_i.sim_size)),
                                tuple(np.ravel(diffraction_i.bloch_vecs)),
                                tuple(np.ravel(diffraction_i.f)[[0, -1]]),
                                len(diffraction_i.f))
                  if theta_orders is None:
                      theta_orders, phi_orders = diffraction_i.angles
                      diffraction_sim_size = tuple(np.ravel(diffraction_i.sim_size))
                      _angle_geometry = geometry_i
                      print(f"Diffraction geometry captured from {filename}: "
                            f"transverse period {diffraction_sim_size} um, "
                            f"{theta_orders.sizes['orders_x']} x {theta_orders.sizes['orders_y']} orders")
                  elif geometry_i != _angle_geometry:
                      raise DiffractionGeometryError(
                          f"Diffraction geometry differs from the stored one in {filename}: "
                          f"{geometry_i} vs {_angle_geometry}. The angle mask is not valid here.")
                 
                  if str(n_value) not in values.keys():
                    values[str(n_value)] = {}
                  if str(ff) not in values[str(n_value)].keys():
                      values[str(n_value)][str(ff)] = {}
                  if str(z_value) not in values[str(n_value)][str(ff)].keys():
                        values[str(n_value)][str(ff)][str(z_value)] = {}
                  if str(size) not in values[str(n_value)][str(ff)][str(z_value)].keys():
                        values[str(n_value)][str(ff)][str(z_value)][str(size)] = {}
                  if str(sample) not in values[str(n_value)][str(ff)][str(z_value)][str(size)].keys():
                        values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)] = {}

                  #p = co and s = cross
                  amps = structure_1.sim_data["diffraction"].amps
                  lambdas = td.C_0 / structure_1.sim_data["diffraction"].f
                  T_total = transmission_exit/reference_exit

                  values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)]["amps"] = amps
                  values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)]["T_total"] = T_total

        except DiffractionGeometryError:
            # must NOT be swallowed by the catch-all below: a wrong angle mask would
            # silently corrupt every aperture quantity downstream
            raise
        except Exception as e:
            print("Error:", e)
            continue

# Fall back to the empty reference if every file was already cached (loop body skipped)
if theta_orders is None:
    diffraction_ref = reference_object.sim_data["diffraction"]
    theta_orders, phi_orders = diffraction_ref.angles
    diffraction_sim_size = tuple(np.ravel(diffraction_ref.sim_size))
    print(f"No new sims loaded; diffraction angles taken from the reference "
          f"(transverse period {diffraction_sim_size} um)")

# Define unconditionally: cell 8 stores `lambdas`, but the in-loop assignment above only
# happens when at least one file is actually read.
lambdas = td.C_0 / np.asarray(theta_orders.f)
       






Could not extract z_value from directory: ../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction
Could not extract z_value from directory: ../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction\n_3.40
Configured successfully.


16:48:14 W. Europe Daylight Time Billed flex credit cost: 3.735.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

Diffraction geometry captured from LSU_ffr_0.2369_size_0.8741258741258742_n_3.40_z_100.0_sample_0.txt: transverse period (11.44, 11.44) um, 11 x 11 orders
Configured successfully.


16:48:19 W. Europe Daylight Time Billed flex credit cost: 3.735.

...

16:59:54 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

Configured successfully.


16:59:59 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [6]:
# Collection aperture at normal incidence: keep the orders inside a cone of numerical
# aperture NA, i.e. sin(theta) <= NA. In vacuum NA = sin(theta_max).
NA_APERTURE = 0.2

if theta_orders is None:
    raise RuntimeError("theta_orders is not defined -- run the loading cell above first.")

# xarray aligns with an INNER join, so a mismatched coordinate silently drops data instead
# of raising. Check the grids agree exactly before relying on that.
def _same_grid(a, b, dims=("f",)):
    return all(np.array_equal(np.asarray(a.coords[d]), np.asarray(b.coords[d])) for d in dims)

if not _same_grid(theta_orders, Pinc):
    raise ValueError("Pinc is on a different frequency grid than the diffraction angles.")

# Evanescent orders carry theta = NaN, which compares False, so they drop out on their own.
# NB: tidy3d's DataArray overrides __eq__ to return a plain bool, so never use `==` for
# elementwise tests here; `<=`, `>`, `&` are the normal xarray ones.
# Unlike an off-axis ring, this cone ALWAYS contains the specular (0,0) order (theta = 0),
# so the ballistic beam is collected too and the cone is never empty.
aperture = np.sin(theta_orders) <= NA_APERTURE

# Angular resolution is set by the transverse period: d(sin theta) = lambda / L. The first
# ring of orders only enters the cone when lambda <= NA * L; above that the cone holds the
# specular order alone, and the result is the specular channel rather than a genuine
# NA-limited measurement. Track the count so this is visible rather than assumed.
Lx_period, Ly_period = diffraction_sim_size
lambda_orders = td.C_0 / theta_orders.f
n_orders_aperture = aperture.sum(dim=("orders_x", "orders_y"))
# Solid angle the retained orders actually cover, vs the ideal cone. When the cone is
# undersampled (one channel) the discrete value overshoots badly -- a diagnostic, not a
# normalization, which is why no per-steradian quantity is stored for this aperture.
omega_aperture = ((lambda_orders**2 / (Lx_period * Ly_period) / np.cos(theta_orders))
                  .where(aperture).sum(dim=("orders_x", "orders_y")))
omega_cone_nominal = 2 * np.pi * (1 - np.sqrt(1 - NA_APERTURE**2))

print(f"NA = {NA_APERTURE} (theta_max = {np.rad2deg(np.arcsin(NA_APERTURE)):.2f} deg): "
      f"orders in the cone min {int(n_orders_aperture.min())}, max {int(n_orders_aperture.max())}; "
      f"{100 * float((np.asarray(n_orders_aperture) < 2).mean()):.1f}% of frequencies see the "
      f"specular order ONLY (first ring enters at lambda <= {NA_APERTURE * Lx_period:.3f} um). "
      f"Covered solid angle {float(omega_aperture.min()):.3f}-{float(omega_aperture.max()):.3f} sr "
      f"vs ideal cone {omega_cone_nominal:.3f} sr")

values_processed = {}
for n in values.keys():
    print(f"Processing n={n}")
    for ff in values[n].keys():
        for z_value in values[n][ff].keys():
            for size in values[n][ff][z_value].keys():
                samples   = list(values[n][ff][z_value][size].keys())
                amps_list = [values[n][ff][z_value][size][s]["amps"] for s in samples]
                T_total_average = np.mean([values[n][ff][z_value][size][s]["T_total"] for s in samples], axis=0)
                N         = len(amps_list)

                for a in amps_list:
                    if not _same_grid(a, theta_orders, dims=("f", "orders_x", "orders_y")):
                        raise ValueError(f"order/frequency grid mismatch at "
                                         f"n={n} ff={ff} z={z_value} size={size}")

                # coherent amplitude average (for the ballistic term) -> DataArray
                amps_average = sum(amps_list) / N

                # incoherent intensity averages ⟨|a|²⟩ -> DataArray (.sel/.sum still work)
                T_co    = sum(abs(a.sel(polarization="p"))**2 for a in amps_list) / N / Pinc
                T_cross = sum(abs(a.sel(polarization="s"))**2 for a in amps_list) / N / Pinc

                T_co_total    = T_co.sum(dim=("orders_x", "orders_y"))
                T_cross_total = T_cross.sum(dim=("orders_x", "orders_y"))

                # what an NA-limited detector on axis collects: coherent beam + near-forward
                # diffuse light. T_co_aperture - T_coh_aperture is the diffuse part collected.
                T_co_aperture    = T_co.where(aperture).sum(dim=("orders_x", "orders_y"))
                T_cross_aperture = T_cross.where(aperture).sum(dim=("orders_x", "orders_y"))
                T_coh_aperture   = ((abs(amps_average.sel(polarization="p"))**2 / Pinc)
                                    .where(aperture).sum(dim=("orders_x", "orders_y")))

                T_ballistic = np.abs(
                    amps_average.sel(orders_x=0, orders_y=0, polarization="p"))**2/ Pinc

                T_ballistic_aperture = T_ballistic.where(aperture).sum(dim=("orders_x", "orders_y"))
                
                if str(n) not in values_processed.keys():
                    values_processed[str(n)] = {}
                if str(ff) not in values_processed[str(n)].keys():
                    values_processed[str(n)][str(ff)] = {}
                if str(z_value) not in values_processed[str(n)][str(ff)].keys():
                      values_processed[str(n)][str(ff)][str(z_value)] = {}
                if str(size) not in values_processed[str(n)][str(ff)][str(z_value)].keys():
                      values_processed[str(n)][str(ff)][str(z_value)][str(size)] = {}
                    
                values_processed[str(n)][str(ff)][str(z_value)][str(size)]["T_ballistic"] = T_ballistic
                values_processed[str(n)][str(ff)][str(z_value)][str(size)]["T_co"] = T_co_total
                values_processed[str(n)][str(ff)][str(z_value)][str(size)]["T_cross"] = T_cross_total
                values_processed[str(n)][str(ff)][str(z_value)][str(size)]["T_total"] = T_total_average
                values_processed[str(n)][str(ff)][str(z_value)][str(size)]["T_co_aperture"] = T_co_aperture
                values_processed[str(n)][str(ff)][str(z_value)][str(size)]["T_cross_aperture"] = T_cross_aperture
                values_processed[str(n)][str(ff)][str(z_value)][str(size)]["T_coh_aperture"] = T_coh_aperture




NA = 0.2 (theta_max = 11.54 deg): orders in the cone min 1, max 5; 83.2% of frequencies see the specular order ONLY (first ring enters at lambda <= 2.288 um). Covered solid angle 0.040-0.489 sr vs ideal cone 0.127 sr
Processing n=3.4


In [7]:
# After the loop, get unique values as arrays
n_unique = np.unique(n_values)
ff_unique = np.unique(ff_values)
size_unique = np.unique(size_values)
z_unique = np.unique(z_values)

In [8]:
values_processed['3.4']['0.2369'].keys()

dict_keys(['100.0', '5.0'])

In [9]:

data = {
    "transmission_data": values_processed,
    "n_values": n_unique,
    "ff_values": ff_unique,
    "size_values": size_unique,
    "z_values": z_unique,
    "lambdas": lambdas,
    # aperture bookkeeping, indexed like the other arrays (ascending frequency)
    "NA_aperture": np.array([NA_APERTURE]),
    "n_orders_aperture": n_orders_aperture,
    "omega_aperture": omega_aperture,
    "omega_cone_nominal": np.array([omega_cone_nominal]),
}

In [10]:
# create_hdf5_from_dict(data,data_path)